# Cat/Dog Challenge

Minimal version: load the saved model, predict the 39 images, draw ROC/PR for cats without the 5 `other` images, and make the 5x8 panel.

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython import get_ipython
from PIL import Image
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from torchvision import transforms
from phys303_hw4_cats_dogs import SmallCNN

get_ipython().run_line_magic("matplotlib", "inline")

img_dir = Path("/Users/jjburrell/Downloads/39-label-images")
out_dir = Path("training_outputs"); out_dir.mkdir(exist_ok=True)
tfm = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
model = SmallCNN(); model.load_state_dict(torch.load("phys303_hw4_outputs/best_model.pt", map_location="cpu")); model.eval()

rows = []
for p in sorted(img_dir.glob("*.jpg")):
    label = "cat" if "cat" in p.stem.lower() else "dog" if "dog" in p.stem.lower() else "other"
    x = tfm(Image.open(p).convert("RGB")).unsqueeze(0)
    prob_dog = torch.sigmoid(model(x)).item()
    rows.append([p.name, str(p), label, 1 - prob_dog, prob_dog, "dog" if prob_dog >= 0.5 else "cat"])

df = pd.DataFrame(rows, columns=["file", "path", "label", "prob_cat", "prob_dog", "pred"])
df.to_csv(out_dir / "challenge_predictions.csv", index=False)

cats = df[df.label != "other"].copy()
y = (cats.label == "cat").astype(int)
fpr, tpr, _ = roc_curve(y, cats.prob_cat)
prec, rec, _ = precision_recall_curve(y, cats.prob_cat)
print("ROC AUC:", round(roc_auc_score(y, cats.prob_cat), 4))
print("AP:", round(average_precision_score(y, cats.prob_cat), 4))

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("ROC for cats")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.tight_layout()
plt.savefig(out_dir / "roc_curve_cats.png", dpi=200)
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(rec, prec)
plt.title("PRC for cats")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.tight_layout()
plt.savefig(out_dir / "precision_recall_curve_cats.png", dpi=200)
plt.show()

fig, axes = plt.subplots(5, 8, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    if i < len(df):
        row = df.iloc[i]
        ax.imshow(Image.open(row["path"]).convert("RGB"))
        ax.set_title(f"label: {row['label']}, pred: {row['pred']}", fontsize=7)
    ax.axis("off")
fig.suptitle("39 images: true label and model prediction", fontsize=14)
fig.tight_layout()
plt.savefig(out_dir / "challenge_panel_5x8.png", dpi=150)
plt.show()

df

ROC AUC: 0.8056
AP: 0.8426


/var/folders/kh/hkkfd68s6mb8b11z27mn1b4h0000gn/T/ipykernel_56116/249537441.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(out_dir / "roc_pr_curves_cats.png", dpi=200); plt.show()
/var/folders/kh/hkkfd68s6mb8b11z27mn1b4h0000gn/T/ipykernel_56116/249537441.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(out_dir / "challenge_panel_5x8.png", dpi=150); plt.show()


,file,path,label,prob_cat,prob_dog,pred
0,AWcat1.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.799023,0.200977,cat
1,AWcat2.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.844489,0.155511,cat
2,AWcat3.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.554130,0.445870,cat
3,AWcat4.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.872239,0.127761,cat
4,AWcat5.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.238903,0.761097,dog
5,AWcat6.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.769246,0.230754,cat
6,AWcat7.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.810207,0.189793,cat
7,AWcat8.jpg,/Users/jjburrell/Downloads/39-label-images/AWc...,cat,0.737895,0.262105,cat
8,AWdog1.jpg,/Users/jjburrell/Downloads/39-label-images/AWd...,dog,0.611187,0.388813,cat
9,AWdog2.jpg,/Users/jjburrell/Downloads/39-label-images/AWd...,dog,0.227393,0.772607,dog


In [ ]:
plot_df = pd.read_csv(out_dir / "challenge_predictions.csv")
samples = plot_df.sample(min(40, len(plot_df)), random_state=0)

fig, axes = plt.subplots(5, 8, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    if i < len(samples):
        row = samples.iloc[i]
        ax.imshow(Image.open(row["path"]).convert("RGB"))
        ax.set_title(f"{row['file']}\n{row['pred']}", fontsize=7)
    ax.axis("off")
fig.suptitle("Challenge Images - Model Predictions", fontsize=14)
fig.tight_layout()
plt.show()
